# 4.3 Lab - Deploying a Simple DAB
Edit the starter bundle in **`04-automated-deployment-dabs/labs/4.3_simple_dab/`** in your editor (VS Code gives YAML autocompletion, see lesson 4.9). Then run the CHECK cell. The solution is in `solutions/4.3_simple_dab/`.

| TODO | File |
|---|---|
| 1. Name the bundle `my_first_bundle` | `databricks.yml` |
| 2. Job `etl_job` with tasks `ingest` → `transform` (dependency) | `resources/etl_job.yml` |
| 3. Job parameter `run_date` (default `""`) | `resources/etl_job.yml` |

In [ ]:
from adelab import bundles
from adelab.spark import PROJECT_ROOT

LAB = PROJECT_ROOT / "04-automated-deployment-dabs" / "labs" / "4.3_simple_dab"
status = bundles.workspace_status()
bundles.tree(LAB)

## CHECK (offline)

In [ ]:
assert not bundles.validate_schema(LAB), "fix the schema errors above"
cfg = bundles.preview(LAB, "dev")
assert cfg["bundle"]["name"] == "my_first_bundle", "TODO 1"
job = cfg["resources"]["jobs"]["etl_job"]
tasks = {t["task_key"]: t for t in job["tasks"]}
assert set(tasks) == {"ingest", "transform"}, "TODO 2: two tasks"
assert tasks["transform"]["depends_on"] == [{"task_key": "ingest"}], "TODO 2: dependency"
assert tasks["ingest"]["notebook_task"]["notebook_path"].endswith("src/ingest.py"), "TODO 2: ingest runs ../src/ingest.py"
assert tasks["transform"]["notebook_task"]["notebook_path"].endswith("src/transform.py"),     "TODO 2: transform runs ../src/transform.py"
assert job.get("parameters") == [{"name": "run_date", "default": ""}], f"TODO 3: job parameters are {job.get('parameters')}"
print("All offline checks passed")

## Deploy and run it (workspace)

In [ ]:
if bundles.require_workspace(status):
    bundles.cli("bundle", "validate", cwd=LAB)
    bundles.cli("bundle", "deploy", cwd=LAB)
    bundles.cli("bundle", "run", "etl_job", "--params", "run_date=2026-10-15", cwd=LAB)

Open the run in the Jobs UI and check that both tasks received `run_date`. Clean up when you're done:

In [ ]:
if bundles.require_workspace(status):
    bundles.cli("bundle", "destroy", "--auto-approve", cwd=LAB)